# 01 — Preprocessing
Pipeline (configurabile in `track3_config.py`):
1. **bandpass** (default 0.5–45 Hz) + notch opzionale
2. **baseline correction** (finestra pre-cue -500..0 ms)
3. **crop** finestra imagined speech (0..2000 ms → 512 campioni)
4. **z-score per canale**, statistiche stimate SOLO sul train del soggetto (no leakage)
5. **resample 256→200 Hz** solo per REVE

Tutto subject-dependent.

In [ ]:
# --- setup: rende importabili i moduli track3_*.py ---
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, matplotlib.pyplot as plt
import track3_config as C, track3_io as io, track3_preproc as P
print(C.summary())
assert C.DATA_ROOT is not None, C._no_data_msg()

## 1. Prima/dopo su una trial

In [ ]:
s=1
tr,_,_ = io.load_subject_all(s)
Xraw = tr.X.copy()
Xpp, t2 = P.preprocess_arrays(tr.X, tr.t, tr.fs)   # filtro+baseline+crop
ch=13  # C3
fig,ax=plt.subplots(2,1,figsize=(10,5),sharex=False)
ax[0].plot(tr.t, Xraw[0,ch]); ax[0].axvline(0,color='k',ls='--'); ax[0].set_title(f'grezzo — canale {tr.clab[ch]}')
ax[1].plot(t2, Xpp[0,ch]); ax[1].set_title('dopo bandpass+baseline+crop'); ax[1].set_xlabel('ms')
plt.tight_layout(); plt.show()

## 2. Pipeline completa subject-dependent (con standardizzazione)

In [ ]:
d = P.preprocess_subject(1)
print('X_train', d['X_train'].shape, '| fs', d['fs'], '| finestra', d['t'][[0,-1]])
print('mean≈0 std≈1:', float(d['X_train'].mean()), float(d['X_train'].std()))

## 3. Feature di banda (per DGCNN) — differential entropy per canale/banda

In [ ]:
bf = P.band_features(d['X_train'], d['fs'])
print('band_features', bf.shape, '=(trials, canali, bande)', list(P.BANDS))

## 4. (Opzionale) salvataggio tensori preprocessati per riuso rapido
Salva un `.pt` per soggetto in `interim/`. Utile se il filtro è lento sulla VM.

In [ ]:
# Scommenta per generare i tensori di tutti i soggetti:
# for s in C.SUBJECTS:
#     out = P.save_subject_tensors(s, tag='bp0.5-45_crop0-2000')
#     print('salvato', out.name)

## 5. Variante per REVE (200 Hz)

In [ ]:
dr = P.preprocess_subject(1, resample_to=C.REVE_FS)
print('REVE X_train', dr['X_train'].shape, '@', dr['fs'], 'Hz')